# Phase 3: Customer Feature Engineering

In this phase, we transform transactional order-level data into 
customer-level features suitable for segmentation and retention modeling.

Objectives:
- Aggregate behavioral metrics at customer level
- Construct RFM (Recency, Frequency, Monetary) features
- Generate additional revenue and engagement metrics
- Prepare final modeling dataset

In [ ]:
# ============================================
# 1. Import Libraries
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# load master df 
# ============================================
# 2. Load Master Dataset
# ============================================

master_df = pd.read_csv(r"C:\Users\hp\Desktop\PROJECTS\customer-segmentation-retention\data\processed\master_dataset.csv")
master_df.head()

In [ ]:
# ============================================
# 2. Datetime Handling
# ============================================

master_df['order_purchase_timestamp'] = pd.to_datetime(
    master_df['order_purchase_timestamp']
)

In [ ]:
# ============================================
# 3. Reference Date
# ============================================

reference_date = master_df['order_purchase_timestamp'].max()
reference_date

## 1. RFM Feature Construction

RFM analysis is a standard customer segmentation technique:

- Recency: How recently a customer made a purchase
- Frequency: How often the customer purchases
- Monetary: How much revenue the customer generates

These features form the foundation of segmentation strategy.

In [ ]:
# ============================================
# 5. Calculate RFM Features
# ============================================

rfm = master_df.groupby('customer_id').agg({

    'order_purchase_timestamp': lambda x: (reference_date - x.max()).days,
    'order_id': 'nunique',
    'payment_value': 'sum'

}).reset_index()

rfm.columns = [
    'customer_id',
    'recency_days',
    'frequency',
    'monetary'
]

rfm.head()

### RFM Feature Meaning

- Lower recency → More recently active customers
- Higher frequency → Loyal / repeat customers
- Higher monetary → High-value customers

In [ ]:
# ============================================
# 5. RFM Distribution
# ============================================

rfm[['recency_days', 'frequency', 'monetary']].describe()

In [ ]:
# ============================================
# 6. Behavioral Metrics
# ============================================

customer_metrics = master_df.groupby('customer_id').agg({

    'payment_value': 'mean',
    'review_score': 'mean'

}).reset_index()

customer_metrics.columns = [
    'customer_id',
    'avg_order_value',
    'avg_review_score'
]

customer_metrics.head()

In [ ]:
# ============================================
# RFM Distribution Overview (Combined View)
# ============================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Recency
axes[0].hist(rfm['recency_days'], bins=40)
axes[0].set_title("Recency Distribution")
axes[0].set_xlabel("Days Since Last Purchase")

# Frequency
axes[1].hist(rfm['frequency'], bins=30)
axes[1].set_title("Frequency Distribution")
axes[1].set_xlabel("Number of Orders")

# Monetary
axes[2].hist(rfm['monetary'], bins=40)
axes[2].set_title("Monetary Distribution")
axes[2].set_xlabel("Total Spending")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# 7. Merge Customer Features
# ============================================

customer_df = pd.merge(
    rfm,
    customer_metrics,
    on="customer_id",
    how="left"
)

customer_df.head()

In [ ]:
# ============================================
# 8. Handle Missing Values
# ============================================

customer_df['avg_review_score'].fillna(
    customer_df['avg_review_score'].median(),
    inplace=True
)

In [ ]:
# ============================================
# 9. Feature Summary
# ============================================

customer_df.describe()

In [ ]:
# ============================================
# 10. Save Feature Dataset
# ============================================

customer_df.to_csv(
    "../data/processed/customer_features.csv",
    index=False
)

print("Feature dataset saved.")

## Feature Engineering Summary

We successfully transformed transactional order data into 
a structured customer-level dataset.

Created features include:
- Recency (days since last purchase)
- Frequency (number of orders)
- Monetary (total revenue)
- Average order value
- Maximum order value
- Average review score

This dataset is now ready for:
- RFM Scoring
- Customer Segmentation (Clustering)
- Churn Prediction Modeling